In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from tqdm import tqdm
import gc

In [ ]:
# Kaggle'daki CSV dosyasını oku
df = pd.read_csv('/kaggle/input/datasets/organizations/nih-chest-xrays/data/Data_Entry_2017.csv')

# 14 temel hastalık sınıfı
Hastaliklar = ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 
               'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 
               'Fibrosis', 'Pleural_Thickening', 'Hernia']

# Multi-label için One-Hot Encoding yapıyoruz
for h in Hastaliklar:
    df[h] = df['Finding Labels'].apply(lambda x: 1 if h in x else 0)

# Veri Sızıntısını Önleme: Hastaları %80 Train - %20 Validation olarak gruplayarak ayırıyoruz
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df['Patient ID']))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"Train Goruntu Sayisi: {len(train_df)}, Validation Goruntu Sayisi: {len(val_df)}")

In [ ]:
# 12 farklı 'images_00X' klasörünün hepsini tarayıp kesin konumları sözlüğe alıyoruz
image_path_dict = {}
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path_dict[file] = os.path.join(root, file)

class NIHRontgenDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        # 12 klasör içinden resmi kesin olarak bulan eski güvenli mantığımız
        img_name = self.df.iloc[idx]['Image Index']
        img_path = image_path_dict.get(img_name)
        
        image = Image.open(img_path).convert('RGB')
        
        hastalik_matrisi = self.df.iloc[idx][Hastaliklar].values.astype(int)
        labels = torch.tensor(hastalik_matrisi, dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, labels

# Standart 224x224 Akademik Klinik Dönüşümleri
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(degrees=10), 
    transforms.ColorJitter(brightness=0.1, contrast=0.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = NIHRontgenDataset(train_df, transform=train_transforms)
val_dataset = NIHRontgenDataset(val_df, transform=val_transforms)

# RAM PATLAMASI ENGELEYİCİ: batch_size=16 ve num_workers=0 (Hafızayı kopyalamaz, asla şişirmez)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=False)

In [ ]:
# Model şasisini çağırıyoruz
weights = models.DenseNet121_Weights.DEFAULT
model = models.densenet121(weights=weights)

# Son katmanı 14 hastalık sınıfına göre ayarlıyoruz
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, 14)

# Cihaz seçimi ve Çift GPU kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"Harika! {torch.cuda.device_count()} GPU aktif. Model iki GPU'ya dagitiliyor.")
    model = nn.DataParallel(model)
else:
    print(f"Tek GPU veya CPU aktif: {device}")

model.to(device)

# --- SINIF DENGESİZLİĞİ ÇÖZÜMÜ ---
pos_weights = []
for hastalik in Hastaliklar:
    positives = train_df[hastalik].sum()
    negatives = len(train_df) - positives
    weight = negatives / (positives + 1e-5)
    pos_weights.append(weight)

pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# VALIDASYONSUZ, ARKA ARKAYA KESİNTİSİZ 12 EPOCH EĞİTİMİ
start_epoch = 0  
total_epochs = 12  

print("=== 12 EPOCHLUK EĞİTİM  ===")

for epoch in range(start_epoch, total_epochs):
    # ==================== EĞİTİM ADIMI ====================
    model.train()
    train_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        
    epoch_train_loss = train_loss / len(train_loader.dataset)

    print(f"\n=== EPOCH {epoch+1} SONUÇLARI ===")
    print(f"Train Loss: {epoch_train_loss:.4f}")
    print("================================\n")
    
    # Her epoch bittiğinde o epoch'un ağırlığını anında diske kilitliyoruz
    torch.save(model.state_dict(), f'densenet121_nih_final_epoch{epoch+1}_224x224.pth')
    print(f"=== BAŞARILI: Epoch {epoch+1} ağırlıkları 'densenet121_nih_final_epoch{epoch+1}_224x224.pth' adıyla kaydedildi! ===\n")
    
    # Hafıza temizliği (Çökme engelleyici)
    torch.cuda.empty_cache()
    gc.collect()